
# 練習問題: 重力 N体シミュレーション (直接法 O(N²))

## 目標

`N` 個の粒子が互いに万有引力を及ぼし合いながら動く様子をシミュレーションする。各粒子に働く力を「他の全粒子からの寄与の和」として計算する二重ループ (O(N²)) が計算の主役で, 並列化・高速化の効果が大きい。マルチコア並列化の典型である「外側ループを `parallel for` で分担する」を体験する。

## 課題

`nbody.cpp` (または `nbody.f90`) は, 各粒子 `i` の加速度を他の全粒子 `j` からの重力の和として求め (`compute_acc`), シンプレクティック・オイラー法で位置・速度を更新する。各粒子 `i` の力の計算は互いに独立なので並列化できる。

`TODO` の箇所に **OpenMP の指示を追加** し, `compute_acc` の外側ループ (粒子 `i` のループ) を並列化せよ。

- C++: 外側 `for` の直前に `#pragma omp parallel for` を1行加える。
- Fortran: 外側 `do` を `!$omp parallel do private(...)` と `!$omp end parallel do` で囲む (各スレッドが使う一時変数を `private` に並べる。既にひな形に書いてある)。

ポイント:

- 並列化するのは**粒子 `i` の外側ループ**。内側の `j` の和は各 `i` が自分の局所変数に積むので, reduction は不要。
- 各 `i` の内側ループの足し算の順序は変わらないので, **スレッド数を変えても結果は完全に同じ**になる。
- 近距離での力の発散を防ぐためソフトニング `eps` を入れている (`1/(r²+eps²)^{3/2}`)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore nbody.cpp -o nbody.exe

# Fortran
nvfortran -fast -mp=multicore nbody.f90 -o nbody.exe
```

第1引数で粒子数 `N`, 第2引数で時間ステップ数。

```
OMP_NUM_THREADS=4 ./nbody.exe 2000 100
```

## 期待される結果

```
N=2000, steps=100: エネルギー -4.635538e-01 -> -4.635649e-01 (相対変化 2.40e-05)
```

- **エネルギー保存**: 全エネルギー (運動 + 位置) は時間が経ってもほぼ一定。相対変化が小さい (1e-4 程度以下) ことが, シミュレーションが正しく動いている証拠になる。シンプレクティック法を使っているので長時間でもエネルギーが暴走しない。
- スレッド数を変えても結果 (エネルギー) は完全に一致する。
- `N` を 2倍にすると計算時間は約 **4倍** になる (O(N²))。ここに `parallel for` が効く。スレッド数を増やして台数効果を測ってみよ (「性能を比べる」セル)。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ nbody.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (初期配置の再現性のため): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* 各粒子 i に働く加速度 = 他の全粒子 j からの重力の和 (直接法, O(N^2))。
   ソフトニング eps で近距離の発散を防ぐ。G=1。
   (j=i の項は dx=0 なので寄与 0。特別扱い不要。) */
void compute_acc(int N, const double * pos, const double * mass, double * acc, double eps) {
  double eps2 = eps * eps;
  // TODO: 各粒子 i のループを #pragma omp parallel for で並列化せよ (i ごとに独立)。
  for (int i = 0; i < N; i++) {
    double xi = pos[3*i], yi = pos[3*i+1], zi = pos[3*i+2];
    double ax = 0.0, ay = 0.0, az = 0.0;
    for (int j = 0; j < N; j++) {
      double dx = pos[3*j]   - xi;
      double dy = pos[3*j+1] - yi;
      double dz = pos[3*j+2] - zi;
      double r2 = dx*dx + dy*dy + dz*dz + eps2;
      double inv = 1.0 / (r2 * sqrt(r2));   /* 1/r^3 (ソフトニング込み) */
      double f = mass[j] * inv;
      ax += f * dx; ay += f * dy; az += f * dz;
    }
    acc[3*i] = ax; acc[3*i+1] = ay; acc[3*i+2] = az;
  }
}

/* 全エネルギー = 運動エネルギー + 位置エネルギー (検算用) */
double energy(int N, const double * pos, const double * vel, const double * mass, double eps) {
  double eps2 = eps * eps, KE = 0.0, PE = 0.0;
  for (int i = 0; i < N; i++) {
    KE += 0.5 * mass[i] * (vel[3*i]*vel[3*i] + vel[3*i+1]*vel[3*i+1] + vel[3*i+2]*vel[3*i+2]);
    for (int j = i + 1; j < N; j++) {
      double dx = pos[3*j]-pos[3*i], dy = pos[3*j+1]-pos[3*i+1], dz = pos[3*j+2]-pos[3*i+2];
      PE -= mass[i] * mass[j] / sqrt(dx*dx + dy*dy + dz*dz + eps2);
    }
  }
  return KE + PE;
}

int main(int argc, char ** argv) {
  int    N     = (argc > 1 ? atoi(argv[1]) : 2000);   /* 粒子数 */
  int    steps = (argc > 2 ? atoi(argv[2]) : 100);    /* 時間ステップ数 */
  double dt = 0.001, eps = 0.05;

  double * pos  = (double *)malloc(sizeof(double) * 3 * N);
  double * vel  = (double *)calloc(3 * N, sizeof(double));
  double * acc  = (double *)malloc(sizeof(double) * 3 * N);
  double * mass = (double *)malloc(sizeof(double) * N);
  /* 初期条件: [-1,1]^3 にランダムに配置, 速度 0, 質量は等しく合計 1。 */
  for (int i = 0; i < N; i++) {
    mass[i] = 1.0 / N;
    pos[3*i]   = 2.0 * draw_rand01(i, 0) - 1.0;
    pos[3*i+1] = 2.0 * draw_rand01(i, 1) - 1.0;
    pos[3*i+2] = 2.0 * draw_rand01(i, 2) - 1.0;
  }

  double E0 = energy(N, pos, vel, mass, eps);
  double t0 = omp_get_wtime();
  for (int t = 0; t < steps; t++) {
    compute_acc(N, pos, mass, acc, eps);
    /* シンプレクティック・オイラー法で時間を進める (v を更新してから x を更新) */
    for (int i = 0; i < 3 * N; i++) vel[i] += acc[i] * dt;
    for (int i = 0; i < 3 * N; i++) pos[i] += vel[i] * dt;
  }
  double elapsed = omp_get_wtime() - t0;
  double E1 = energy(N, pos, vel, mass, eps);

  printf("N=%d, steps=%d: エネルギー %.6e -> %.6e (相対変化 %.2e)\n",
         N, steps, E0, E1, fabs((E1 - E0) / E0));
  printf("elapsed = %.3f sec\n", elapsed);
  free(pos); free(vel); free(acc); free(mass);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore nbody.cpp -o nbody_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./nbody_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ nbody.f90
module nbody_mod
contains
  ! 状態を持たない乱数 (初期配置の再現性のため): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! 各粒子 i に働く加速度 = 他の全粒子 j からの重力の和 (直接法 O(N^2))。
  ! ソフトニング eps で近距離の発散を防ぐ。G=1。(j=i の項は dx=0 なので寄与 0。)
  subroutine compute_acc(N, pos, mass, acc, eps)
    integer, intent(in) :: N
    real(8), intent(in) :: pos(3,N), mass(N), eps
    real(8), intent(out) :: acc(3,N)
    real(8) :: eps2, xi, yi, zi, ax, ay, az, dx, dy, dz, r2, inv, f
    integer :: i, j
    eps2 = eps * eps
    ! TODO: 各粒子 i のループを !$omp parallel do (i ごとに独立) で並列化せよ。
    do i = 1, N
       xi = pos(1,i); yi = pos(2,i); zi = pos(3,i)
       ax = 0.0d0; ay = 0.0d0; az = 0.0d0
       do j = 1, N
          dx = pos(1,j) - xi; dy = pos(2,j) - yi; dz = pos(3,j) - zi
          r2 = dx*dx + dy*dy + dz*dz + eps2
          inv = 1.0d0 / (r2 * sqrt(r2))      ! 1/r^3 (ソフトニング込み)
          f = mass(j) * inv
          ax = ax + f*dx; ay = ay + f*dy; az = az + f*dz
       end do
       acc(1,i) = ax; acc(2,i) = ay; acc(3,i) = az
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end subroutine compute_acc

  ! 全エネルギー = 運動エネルギー + 位置エネルギー (検算用)
  function energy(N, pos, vel, mass, eps) result(E)
    integer, intent(in) :: N
    real(8), intent(in) :: pos(3,N), vel(3,N), mass(N), eps
    real(8) :: E, eps2, KE, PE, dx, dy, dz
    integer :: i, j
    eps2 = eps * eps; KE = 0.0d0; PE = 0.0d0
    do i = 1, N
       KE = KE + 0.5d0 * mass(i) * (vel(1,i)**2 + vel(2,i)**2 + vel(3,i)**2)
       do j = i + 1, N
          dx = pos(1,j)-pos(1,i); dy = pos(2,j)-pos(2,i); dz = pos(3,j)-pos(3,i)
          PE = PE - mass(i) * mass(j) / sqrt(dx*dx + dy*dy + dz*dz + eps2)
       end do
    end do
    E = KE + PE
  end function energy
end module nbody_mod

program nbody
  use nbody_mod
  use omp_lib
  character(len=32) :: arg
  integer :: N, steps, t, i
  real(8) :: dt, eps, E0, E1, t0, elapsed
  real(8), allocatable :: pos(:,:), vel(:,:), acc(:,:), mass(:)
  N = 2000; steps = 100; dt = 0.001d0; eps = 0.05d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) steps
  end if

  allocate(pos(3,N), vel(3,N), acc(3,N), mass(N))
  vel = 0.0d0
  ! 初期条件: [-1,1]^3 にランダムに配置, 速度 0, 質量は等しく合計 1。
  do i = 1, N
     mass(i) = 1.0d0 / N
     pos(1,i) = 2.0d0 * draw_rand01(int(i-1,8), 0_8) - 1.0d0
     pos(2,i) = 2.0d0 * draw_rand01(int(i-1,8), 1_8) - 1.0d0
     pos(3,i) = 2.0d0 * draw_rand01(int(i-1,8), 2_8) - 1.0d0
  end do

  E0 = energy(N, pos, vel, mass, eps)
  t0 = omp_get_wtime()
  do t = 1, steps
     call compute_acc(N, pos, mass, acc, eps)
     ! シンプレクティック・オイラー法 (v を更新してから x を更新)
     vel = vel + acc * dt
     pos = pos + vel * dt
  end do
  elapsed = omp_get_wtime() - t0
  E1 = energy(N, pos, vel, mass, eps)

  print "(a,i0,a,i0,a,es13.6,a,es13.6,a,es9.2,a)", &
       "N=", N, ", steps=", steps, ": エネルギー ", E0, " -> ", E1, &
       " (相対変化 ", abs((E1 - E0) / E0), ")"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program nbody

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore nbody.f90 -o nbody_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./nbody_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:nbody.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:nbody.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:nbody.cpp}